In [ ]:
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
import pathlib
from pathlib import Path

First, I wrote two functions to compile all the calibration files into a dictionary for the corresponding observation.
From previous attempts, I've observed that for all the files of observation 2507, the data is in hdul[0], while for 2505 and 2508, the data is in hdul[0] for all except the raw.fits and blevfile.fits, where the data is in hdul[1].

In [ ]:
def load_data(file):
"""
Takes input the fits file, and return the data,
primary header, and data header

Parameters:
-----------
file: fits file
    file to extract the data and headers from

Returns:
-----------
data: np.array
    data extracted
pheader: header
    primary header
dheader: header
    data header
"""

    with fits.open(file) as hdul:
        pheader = hdul[0].header

        # For the already calibrated files, only need to return
        # the data
        if str(file).endswith("aligned.fits"):
            data = hdul[1].data
            return data
        # 2507 is structured differently, so for this observation
        # the data is from the the first header
        elif '2507' in str(file):
            data = hdul[0].data
        # raw and blev data is in header 1
        elif str(file).endswith("raw.fits") or str(file).endswith("blevfile.fits"):
            data = hdul[1].data
        else:
            data = hdul[0].data
        dheader = hdul[1].header
    return data, pheader, dheader

In [ ]:
def load_file(filelist):
"""
Take the list of files, and build the dictionary for easy access

Parameters:
-----------
filelist: path
    path item for the calibration files
    

Returns:
-----------
file_dict: dictionary
    dictionary containing all the data
    headers for each of the calibration
    files in the directory
"""

    file_dict = {}
    
    for file in filelist:

        # For each file in the file list, we call the load_data function
        data, pheader, dheader = load_data(file)
        file_dict[str(file)] = (data, pheader, dheader)
    return file_dict

I then wrote two functions to handle the blev file correction, as it requires a bit more preprocessing then the other files.

In [ ]:
def blev_parsing(data):
"""
Takes the blev data in, and returns the mean values
of the even and odd overscan regions

Parameters:
-----------
data: np.array
     blev data to parse
    

Returns:
-----------
np.mean(oddv), np.mean(evenv): float
    the mean value of the even and odd regions
"""
    # Define the area of interest of the data
    area = data[9:790, 8:14]
    oddv = area[:, 0::2]
    evenv = area[:, 1::2]
    return np.mean(oddv), np.mean(evenv)

In [ ]:
def apply_bias(raw, blev):
"""
Takes the raw and blev data, and return the adjusted
data 

Parameters:
-----------
raw, blev: np.array
    data array for the raw and blev files
    

Returns:
-----------
corr_data: np.array
    data array with applied blev bias correction
"""

    # Make a copy of the raw data
    corr_data = raw.astype('float64')
    
    # Multi chip case
    if raw.ndim == 3 and blev.ndim == 3 and blev.shape[-1] == 14 and raw.shape[0] == blev.shape[0]:
        for c in range(raw.shape[0]):

            # Load mean odds and evens from blev_parsing
            mean_o, mean_e = blev_parsing(blev[c])

            # Subtract the means from the data
            corr_data[c, :, 0::2] -= mean_o
            corr_data[c, :, 1::2] -= mean_e
        return corr_data

    # Single chip case  
    mean_o, mean_e = blev_parsing(blev)
    corr_data[:, 0::2] -= mean_o
    corr_data[:, 1::2] -= mean_e
    return corr_data

Once the blev file handling is done, I proceeded to write a function to perform all the calibration steps highlighted in the assignment brief:
\
stars + noise = (raw - bias - dark) / flat - sky

In [ ]:
def calibration(dpath):
"""
Takes the paths to the fits files, and return the final_cal
calibrated file and exposure time

Parameters:
-----------
dpath: path
    path item the calibration files

Returns:
-----------
final_cal: np.array
    calibrated data array
exptime: float
    observation exposure time 
"""

    # Create the corresponding dictionary with the load_file function
    files = load_file(dpath)
    for filename, file_tuple in files.items():

        # Cycle through the files
        if 'raw' in filename:
            raw_data = file_tuple[0]
            exptime = file_tuple[1]['EXPTIME']
        elif 'bias' in filename:
            bias_data = file_tuple[0]
        # Decided to ignore the mask data
        #elif 'mask' in filename:
        #    mask_data = file_tuple[0]
        elif 'blev' in filename:
            blev_data = file_tuple[0]
        elif 'dark' in filename:
            dark_data = file_tuple[0]
        elif 'flat' in filename:
            flat_data = file_tuple[0]

    # Apply all the calibration steps as described in the brief
    corr_blev = apply_bias(raw_data, blev_data)
    corr_bias = corr_blev - bias_data
    # Scaling the dark_data for the exposure time
    dark_scaled = dark_data * exptime
    corr_dark = corr_bias - dark_scaled
    corr_flat = corr_dark * flat_data
    #bad_pixels = mask_data > 0
    #corr_flat[bad_pixels] = np.nan
    sky_bkg = np.nanmedian(corr_flat)
    final_cal = corr_flat - sky_bkg
    return final_cal, exptime

With the output of the calibration, I wrote then a function to apply the photometry corrections

In [ ]:
def photom(aligned_file,exptime, photflam = 1.115396e-16,
           photzpt = -21.1, zp_corr = -0.03288031):
"""
Takes the calibrated file and exposure time, and returns
flux and magnitude

Parameters:
-----------
aligned_file: np.array
    calibrated data array
exptime: float
    observation exposure time
photflam: float
    inverse sensitivity
photzpt: float
    zero point
zp_corr: float
    delta magnitude correction for contamination

Returns:
-----------
flux: np.array
    flux value data array
mag: np.array
    magnitude value data array
"""

    # Calculate the rate for flux and magnitude
    rate = aligned_file / exptime
    flux = rate * photflam
    mag = (-2.5 * np.log10(rate * photflam) + photzpt) + zp_corr
    return flux, mag

With all the calibration done, I wrote functions to check the calibrated data resulting from my pipeline with the aligned ones, by plotting them side by side. One function checks the simple calibrated images, while the other compares the photometry

In [ ]:
def calib_check(og, my, title, outdir):
"""
Takes the aligned and calibrated files, the plot title
and directory, and save the comparative plots in the
specified directory 

Parameters:
-----------
og: np.array
    aligned data array
my: np.array
    pipeline calibratted data array
title: string
    title of the obserbvation
outdir: string
    name of the output directory

Returns:
-----------
saved plot of the comparison
"""

    # Assign the output directory
    outdir = Path(outdir)
    fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(10, 5))

    # Plot the pipeline-calibrated and aligned images next to each other
    for ax, img, name in ((ax1, og, "Aligned file"), (ax2, my, "Calibrated file")):

        # compute the vmin and vmax value from the data
        vmin, vmax = np.nanpercentile(img, [1, 99])
        ax.imshow(img, origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
        ax.set_title(name)
    fig.suptitle(title)
    plt.show()

    # Save the plot to the directory
    fig.savefig(outdir / f"Calib_check_{title}.jpg", dpi=300, bbox_inches='tight')

In [ ]:
def flux_check(og, my, title, outdir):
"""
Takes the aligned and calibrated files, the plot title
and directory, and save the comparative flux plots in the
specified directory 

Parameters:
-----------
og: np.array
    aligned data array
my: np.array
    pipeline calibratted data array
title: string
    title of the obserbvation
outdir: string
    name of the output directory

Returns:
-----------
saved plot of the comparison
"""

    # Assign the output directory
    outdir = Path(outdir)
    fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(11, 5), constrained_layout=True)

    # Plot the pipeline-calibrated and aligned images next to each other
    for ax, img, name in ((ax1, og, "Aligned file"), (ax2, my, "Calibrated file")):
        
        # compute the vmin and vmax value from the data
        vmin, vmax = np.nanpercentile(img, [1, 99])
        im = ax.imshow(img, origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
        ax.set_title(name)

        # Add the colorbar for the flux 
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(r"Flux [erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$]")
    fig.suptitle(title)
    plt.show()

    # Save the plot to the directory
    fig.savefig(outdir / f"Flux_check_{title}.jpg", dpi=300, bbox_inches='tight')

Once all the functions were written down, I proceeded to call them in order

In [ ]:
# Build the file paths
here = pathlib.Path('.')
ubai2505m = list(here.rglob('**/ubai2505m/**/*.fits'))
ubai2507m = list(here.rglob('**/ubai2507m/**/*.fits'))
ubai2508m = list(here.rglob('**/ubai2508m/**/*.fits'))

ubai2505m_og_path = 'hst-original/ubai2505m_c0m_aligned.fits'
ubai2507m_og_path = 'hst-original/ubai2507m_c0m_aligned.fits'
ubai2508m_og_path = 'hst-original/ubai2508m_c0m_aligned.fits'

In [ ]:
# Call the calibration functions
ubai2505m_my_cal, ubai2505m_my_cal_exptime = calibration(ubai2505m)
ubai2507m_my_cal, ubai2507m_my_cal_exptime = calibration(ubai2507m)
ubai2508m_my_cal, ubai2508m_my_cal_exptime = calibration(ubai2508m)
ubai2505m_og_cal = load_data(ubai2505m_og_path)
ubai2507m_og_cal = load_data(ubai2507m_og_path)
ubai2508m_og_cal = load_data(ubai2508m_og_path)

In [ ]:
# Compute the flux for the calibrated and aligned files
ubai2505m_my_flux, ubai2505m_my_mag = photom(ubai2505m_my_cal, ubai2505m_my_cal_exptime)
ubai2505m_og_flux, ubai2505m_og_mag = photom(ubai2505m_og_cal, ubai2505m_my_cal_exptime)
ubai2507m_my_flux, ubai2507m_my_mag = photom(ubai2507m_my_cal, ubai2507m_my_cal_exptime)
ubai2507m_og_flux, ubai2507m_og_mag = photom(ubai2507m_og_cal, ubai2507m_my_cal_exptime)
ubai2508m_my_flux, ubai2508m_my_mag = photom(ubai2508m_my_cal, ubai2508m_my_cal_exptime)
ubai2508m_og_flux, ubai2508m_og_mag = photom(ubai2508m_og_cal, ubai2508m_my_cal_exptime)

In [ ]:
# Comparison of the calibration (aligned vs pipeline calibration)
calib_check(ubai2505m_og_cal, ubai2505m_my_cal[0], "Observation_ubai2505", "hst-processed")
calib_check(ubai2507m_og_cal, ubai2507m_my_cal[0], "Observation_ubai2507", "hst-processed")
calib_check(ubai2508m_og_cal, ubai2508m_my_cal[0], "Observation_ubai2508", "hst-processed")

# Comparison of the flux (aligned vs pipeline calibration)
flux_check(ubai2505m_og_flux, ubai2505m_my_flux[0], "Observation_ubai2505", "hst-processed")
flux_check(ubai2507m_og_flux, ubai2507m_my_flux[0], "Observation_ubai2507", "hst-processed")
flux_check(ubai2508m_og_flux, ubai2508m_my_flux[0], "Observation_ubai2508", "hst-processed")